# Дополнительный модуль: источники данных, строки, регулярные выражения и файлы

Этот ноутбук продолжает тему **«Сбор данных с различных источников. Основы работы с SQL»**.

В первых файлах мы работали с MySQL: создавали таблицы `users` и `orders`, писали SQL-запросы, соединяли таблицы через `JOIN`. Но в реальной работе аналитик получает данные не только из базы данных. Данные часто приходят из разных источников:

- CSV-файлов;
- Excel-файлов;
- JSON-файлов;
- текстовых полей;
- выгрузок из CRM, сайтов, форм и внутренних систем.

Цель этого ноутбука — показать, как после получения данных привести их в порядок: проверить типы, очистить строки, извлечь нужные части текста, объединить таблицы и сохранить результат в разные форматы.

## Что слушатель должен понять

После прохождения ноутбука слушатель должен уметь:

- отличать числовой тип данных от строкового;
- преобразовывать данные через `astype()`;
- работать со строковыми столбцами через `.str`;
- фильтровать email, URL и текстовые записи;
- извлекать части строки через регулярные выражения;
- добавлять строки в `Series` и `DataFrame` через `pd.concat()`;
- строить простые графики;
- создавать `DataFrame` из словаря и списка словарей;
- читать и записывать CSV, Excel и JSON;
- понимать, где такие операции встречаются после SQL-выгрузки.

## 1. Подготовка библиотек

Нам понадобятся:

- `pandas` — для таблиц;
- `matplotlib` — для простых графиков;
- `Path` из `pathlib` — чтобы удобно работать с папками и файлами.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:,.2f}'.format

DATA_DIR = Path('data_sources_demo')
DATA_DIR.mkdir(exist_ok=True)

print('Папка для учебных файлов:', DATA_DIR.resolve())

## 2. Типы данных: число, строка и обратное преобразование

В SQL мы уже видели разные типы данных:

```sql
age INT,
balance DECIMAL(10, 2),
email VARCHAR(50)
```

В pandas у каждого столбца тоже есть тип данных. Это важно, потому что число и строка ведут себя по-разному.

Например, число `25` можно сравнивать и складывать. А строка `'25 лет'` уже не число: её сначала нужно очистить и преобразовать обратно.

In [ ]:
users_age = pd.DataFrame({
    'name': ['Ivan', 'Gleb', 'Sergey'],
    'age': [25, 26, 28]
})

print('Исходная таблица:')
display(users_age)
print('\nТипы данных:')
print(users_age.dtypes)

Добавим текстовое представление возраста. Такая ситуация часто встречается в «грязных» выгрузках: вместо числа в ячейке может быть строка вида `25 лет`.

In [ ]:
users_age['age_text'] = users_age['age'].astype(str) + ' лет'

display(users_age)
print(users_age.dtypes)

Теперь восстановим число из строки:

1. уберём текст ` лет`;
2. преобразуем результат в `int`.

In [ ]:
users_age['age_recovered'] = (
    users_age['age_text']
    .str.replace(' лет', '', regex=False)
    .astype(int)
)

display(users_age)
print(users_age.dtypes)

### Мини-задание 1

Создайте таблицу `salary_df` с колонкой `salary` и значениями `[145000, 152000, 161000]`.

Затем:

1. создайте новую колонку `salary_text` в формате `145000 руб.`;
2. восстановите из неё числовую колонку `salary_recovered`;
3. проверьте типы данных.

In [ ]:
# Место для самостоятельной работы слушателя

## 3. Строковые методы: email и домены

В базах данных часто есть поля с email, телефонами, адресами сайтов, комментариями. Формально это строки, но внутри строки есть полезная структура.

Например, по email можно понять домен: `example.com`, `example.org`, `company.ru`.

In [ ]:
emails = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5, 6],
    'email': [
        'ivan@example.com',
        'gleb@example.com',
        'sergey@example.org',
        'andrew@company.ru',
        'bob@example.com',
        'tom@company.ru'
    ]
})

display(emails)

Отфильтруем строки, где email заканчивается на `.com`.

Здесь используется регулярное выражение `\.com$`:

- `\.` означает точку как обычный символ;
- `com` — текст `com`;
- `$` — конец строки.

In [ ]:
mask_com = emails['email'].str.contains(r'\.com$', regex=True)

print('Булева маска:')
print(mask_com)

print('\nТолько email с доменом .com:')
display(emails[mask_com])

Теперь извлечём домен после символа `@`.

In [ ]:
emails['domain'] = emails['email'].str.extract(r'@(.+)$')
display(emails)

### Мини-задание 2

Создайте таблицу `urls` с колонкой `url`:

```python
['https://site.org', 'http://example.com', 'https://docs.org', 'https://shop.ru']
```

Затем:

1. отфильтруйте адреса, которые заканчиваются на `.org`;
2. создайте колонку `is_https`, где будет `True`, если адрес начинается с `https`;
3. извлеките доменную зону: `org`, `com`, `ru`.

In [ ]:
# Место для самостоятельной работы слушателя

## 4. Извлечение данных из текстовых записей

Иногда данные приходят не в нормальной таблице, а в одном текстовом поле.

Например:

```text
id=105; user=Alice; score=92
```

В такой строке спрятаны три разных признака: `id`, `user`, `score`.

In [ ]:
records = pd.DataFrame({
    'record': [
        'id=105; user=Alice; score=92',
        'id=106; user=Bob; score=85',
        'id=107; user=Charlie; score=97',
        'id=108; user=Diana; score=78'
    ]
})

display(records)

Извлечём значения через `.str.extract()`.

Регулярное выражение ниже содержит группы в круглых скобках:

- `(\d+)` — одна или несколько цифр;
- `([A-Za-z]+)` — одно или несколько латинских букв.

In [ ]:
records[['id', 'user', 'score']] = records['record'].str.extract(
    r'id=(\d+); user=([A-Za-z]+); score=(\d+)'
)

records['id'] = records['id'].astype(int)
records['score'] = records['score'].astype(int)

display(records)
print(records.dtypes)

Теперь можно анализировать извлечённые значения как обычные столбцы.

In [ ]:
print('Средний score:', records['score'].mean())
print('Максимальный score:', records['score'].max())

print('\nПользователи со score >= 90:')
display(records[records['score'] >= 90])

### Мини-задание 3

Создайте таблицу с колонкой `raw_order`:

```python
[
    'order=1; product=Book; qty=2',
    'order=2; product=Mouse; qty=1',
    'order=3; product=Keyboard; qty=3'
]
```

Извлеките в отдельные колонки:

- `order_id`;
- `product`;
- `qty`.

Колонки `order_id` и `qty` должны быть числовыми.

In [ ]:
# Место для самостоятельной работы слушателя

## 5. Добавление данных через `pd.concat()`

В старых примерах pandas часто можно встретить добавление строк через `append()`. В современных версиях pandas для добавления строк и объединения таблиц используют `pd.concat()`.

Смысл простой: мы не «дописываем» строку в таблицу вручную, а объединяем две таблицы или две серии.

In [ ]:
scores = pd.Series([5, 10, 15], name='score')
print('Исходная Series:')
print(scores)

In [ ]:
scores = pd.concat([
    scores,
    pd.Series([20, 25], name='score')
], ignore_index=True)

print('После добавления значений в конец:')
print(scores)

In [ ]:
scores = pd.concat([
    pd.Series([0], name='score'),
    scores
], ignore_index=True)

print('После добавления значения в начало:')
print(scores)

Теперь добавим строки в `DataFrame`.

In [ ]:
clients = pd.DataFrame({
    'name': ['Ivan', 'Gleb', 'Sergey'],
    'age': [25, 26, 28],
    'country': ['RUS', 'RUS', 'RUS']
})

new_clients = pd.DataFrame([
    {'name': 'Andrew', 'age': 24, 'country': 'RUS'},
    {'name': 'Tom', 'age': 29, 'country': 'USA'}
])

clients = pd.concat([clients, new_clients], ignore_index=True)
display(clients)

### Мини-задание 4

Создайте таблицу `products` с колонками:

- `product_name`;
- `price`;
- `quantity`.

Затем создайте ещё одну таблицу `new_products` с двумя новыми товарами и объедините их через `pd.concat()`.

In [ ]:
# Место для самостоятельной работы слушателя

## 6. Простые графики

График помогает быстро увидеть динамику или сравнение.

На этом этапе нам не нужно строить сложную визуализацию. Достаточно понять принцип:

- линейный график — хорошо показывает динамику;
- столбчатая диаграмма — хорошо сравнивает категории.

In [ ]:
daily_revenue = pd.Series(
    [20, 65, 45, 24, 25],
    index=pd.to_datetime(['2023-02-20', '2023-02-22', '2023-02-23', '2023-02-25', '2023-02-26']),
    name='revenue'
)

display(daily_revenue)

In [ ]:
ax = daily_revenue.plot(kind='line', marker='o', title='Выручка по дням')
plt.xlabel('Дата')
plt.ylabel('Выручка')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
product_revenue = pd.DataFrame({
    'product_name': ['Product A', 'Product B', 'Product C', 'Product D', 'Product E'],
    'revenue': [20, 20, 45, 24, 25]
})

ax = product_revenue.plot(x='product_name', y='revenue', kind='bar', title='Выручка по товарам')
plt.xlabel('Товар')
plt.ylabel('Выручка')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Мини-задание 5

Создайте `Series` со значениями `[2, 4, 1, 5, 3]` и постройте линейный график.

Затем создайте `DataFrame` с тремя колонками `sales_A`, `sales_B`, `sales_C` и постройте столбчатую диаграмму.

In [ ]:
# Место для самостоятельной работы слушателя

## 7. Словари и DataFrame

В Python словарь часто похож на одну запись из JSON или одну строку из внешней системы.

Например, товар может быть представлен так:

In [ ]:
product = {
    'name': 'Keyboard',
    'price': 3500,
    'qty': 2
}

print(product)

Словарь можно обновлять: менять существующие значения и добавлять новые ключи.

In [ ]:
product['price'] = 3700
product['in_stock'] = True

for key, value in product.items():
    print(key, '->', value)

Список словарей легко превращается в таблицу `DataFrame`.

In [ ]:
cities = [
    {'city': 'Moscow', 'population_mln': 12.6},
    {'city': 'Saint Petersburg', 'population_mln': 5.6},
    {'city': 'Kazan', 'population_mln': 1.3}
]

cities_df = pd.DataFrame(cities)
cities_df = cities_df.set_index('city').sort_values('population_mln', ascending=False)
display(cities_df)

### Мини-задание 6

Создайте словарь `order` с полями:

- `product`;
- `price`;
- `qty`.

Затем:

1. измените цену;
2. добавьте поле `status`;
3. выведите все ключи и значения;
4. создайте `DataFrame` из списка таких словарей.

In [ ]:
# Место для самостоятельной работы слушателя

## 8. CSV: самый частый формат обмена таблицами

CSV — это текстовый файл, где строки таблицы записаны построчно, а значения разделены символом-разделителем, чаще всего запятой или точкой с запятой.

CSV часто используют для обмена данными между системами, BI-инструментами и аналитиками.

In [ ]:
students = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [24, 30, 40],
    'course': ['DA', 'BI', 'Python'],
    'grade': ['A', 'B', 'A']
})

display(students)

In [ ]:
csv_path = DATA_DIR / 'students.csv'

students.to_csv(csv_path, index=False, encoding='utf-8')
print('CSV-файл записан:', csv_path)

In [ ]:
students_from_csv = pd.read_csv(csv_path, encoding='utf-8')

display(students_from_csv)
print(students_from_csv.dtypes)

### Важное замечание про CSV

Если файл открывается в Excel некорректно, проблема часто не в pandas, а в:

- кодировке;
- разделителе;
- региональных настройках Excel.

Для русскоязычных Excel-файлов иногда используют `sep=';'` и `encoding='utf-8-sig'`.

In [ ]:
csv_semicolon_path = DATA_DIR / 'students_semicolon.csv'

students.to_csv(csv_semicolon_path, index=False, sep=';', encoding='utf-8-sig')
print('CSV с разделителем ; записан:', csv_semicolon_path)

## 9. Excel: привычный формат для пользователей бизнеса

Excel удобен для передачи таблиц между людьми, но для автоматической обработки данных его нужно читать аккуратно:

- указывать лист;
- проверять заголовки;
- проверять типы данных;
- следить за пустыми строками.

In [ ]:
excel_path = DATA_DIR / 'students.xlsx'

try:
    students.to_excel(excel_path, index=False, sheet_name='Students')
    print('Excel-файл записан:', excel_path)
except Exception as error:
    print('Не удалось записать Excel-файл.')
    print('Возможная причина: не установлен пакет openpyxl.')
    print('Ошибка:', repr(error))

In [ ]:
try:
    students_from_excel = pd.read_excel(excel_path, sheet_name='Students')
    display(students_from_excel)
    print(students_from_excel.dtypes)
except Exception as error:
    print('Не удалось прочитать Excel-файл.')
    print('Возможная причина: не установлен пакет openpyxl.')
    print('Ошибка:', repr(error))

## 10. JSON: формат для обмена между приложениями

JSON часто используется в API, веб-сервисах и внутренних системах.

Для аналитика важно понимать, что JSON может быть:

- списком объектов;
- вложенной структурой;
- одной записью;
- набором строк, где каждая строка — отдельный JSON-объект.

В этом ноутбуке используем простой вариант: список объектов.

In [ ]:
json_path = DATA_DIR / 'students.json'

students.to_json(json_path, orient='records', force_ascii=False, indent=2)
print('JSON-файл записан:', json_path)

In [ ]:
students_from_json = pd.read_json(json_path, orient='records')

display(students_from_json)
print(students_from_json.dtypes)

## 11. Полный мини-пайплайн: прочитали, очистили, сохранили

Теперь соберём небольшой пример, похожий на реальную работу аналитика:

1. получили данные в CSV;
2. прочитали их в pandas;
3. очистили строковые поля;
4. извлекли домен email;
5. рассчитали новый столбец;
6. сохранили результат в CSV и JSON.

In [ ]:
raw_clients = pd.DataFrame({
    'client_id': [1, 2, 3, 4],
    'name': ['  Ivan ', ' Gleb', 'Sergey  ', '  Andrew'],
    'email': ['ivan@example.com', 'gleb@example.com', 'sergey@example.org', 'andrew@company.ru'],
    'balance_text': ['100000 руб.', '110000 руб.', '90000 руб.', '95000 руб.']
})

raw_clients_path = DATA_DIR / 'raw_clients.csv'
raw_clients.to_csv(raw_clients_path, index=False, encoding='utf-8')
print('Исходная выгрузка записана:', raw_clients_path)
display(raw_clients)

In [ ]:
clients_raw = pd.read_csv(raw_clients_path, encoding='utf-8')

display(clients_raw)
print(clients_raw.dtypes)

In [ ]:
clients_clean = clients_raw.copy()

clients_clean['name'] = clients_clean['name'].str.strip()
clients_clean['email_domain'] = clients_clean['email'].str.extract(r'@(.+)$')
clients_clean['balance'] = (
    clients_clean['balance_text']
    .str.replace(' руб.', '', regex=False)
    .astype(int)
)
clients_clean = clients_clean.drop(columns=['balance_text'])

display(clients_clean)
print(clients_clean.dtypes)

In [ ]:
clean_csv_path = DATA_DIR / 'clients_clean.csv'
clean_json_path = DATA_DIR / 'clients_clean.json'

clients_clean.to_csv(clean_csv_path, index=False, encoding='utf-8')
clients_clean.to_json(clean_json_path, orient='records', force_ascii=False, indent=2)

print('Очищенный CSV:', clean_csv_path)
print('Очищенный JSON:', clean_json_path)

## 12. Итоговая практика

Выполните задания ниже самостоятельно. Они повторяют основные операции ноутбука.

### Задания

1. Создайте `DataFrame` с колонкой `Salary` и значениями `[145000, 152000, 161000]`.
2. Создайте колонку `SalaryText` в формате `145000 руб.`.
3. Извлеките обратно числовую колонку `SalaryRecovered`.
4. Создайте таблицу `URL` и отфильтруйте строки, которые заканчиваются на `.org`.
5. Извлеките из строк вида `id=105; user=Alice; score=92` пользователя и балл.
6. Добавьте значения в `Series` через `pd.concat()`.
7. Добавьте две строки в `DataFrame` через `pd.concat()`.
8. Постройте линейный график по числовой `Series`.
9. Создайте список словарей и преобразуйте его в `DataFrame`.
10. Сохраните итоговую таблицу в CSV и JSON.

In [ ]:
# Итоговая практика: место для работы слушателя